# Data Merging — Stage 4 Assembly 05: Splits

## Input
- `Data/Data_Collection/Final/Stage_5_Model_Ready/02_assembled/{agg_means, agg_full_moments, panel}.parquet` (model-ready, clipped/filled feature tables from notebooks 02/03)
- `Data/Data_Collection/Final/Stage_5_Model_Ready/03_targets/{targets_market, targets_panel}.parquet` (from notebook 04)
- `Data/Data_Collection/Final/Stage_5_Model_Ready/03_targets/huber_delta.json` (from notebook 04)
- `lib.config` — `OUT`, `CLIP`, `DTYPE`, `CRASH_THRESH`, `BINARIES`

## Purpose
The final assembly step: merges targets onto each of the three model-ready feature tables (`agg_means`, `agg_full_moments`, `panel`), carves each into four time-based train/val/test splits with an embargo to prevent target leakage across split boundaries, validates the splits exhaustively, and writes out the final per-split files plus a metadata manifest that downstream training code reads directly.

## Cell 1 — Load and Merge Targets
For each of the three datasets, merges the relevant target table onto the assembled feature table via an **inner join** on `date` (or `permno`+`date` for the panel). The inner join is deliberate: rows without a complete forward-looking target window (handled in notebook 04) have no target value and simply drop out here rather than needing an explicit filter.

The market-side merge drops `target_daily_return` from the *feature* table before merging (it's re-added via the targets file, since keeping two copies would collide), and the panel-side merge drops `dlyret` from the targets file (it's already present in the feature table).

Columns are explicitly reordered so **meta columns come first, features follow** (`META[name] + [remaining columns]`), specifically so downstream loading code can do `df[feature_cols]` without needing any column lookup logic. Asserts zero NaN remains in the feature columns after merging.

## Cell 2 — Build Splits, With Embargo

### The Splits
Four fixed splits (`SPLIT_A` through `SPLIT_D`), each with a named market regime description (pre-COVID stress, COVID crash/recovery, rate-hiking bear market — marked "Primary" — and the AI-rally recovery period with the most training data). Explicitly noted that `Split_B`'s `train_end` (2017-12-31) is unchanged from the earlier pipeline stages **on purpose**, because that same date was already used as the diagnostic window boundary to freeze the feature set back in Stage 3 — changing it here would create an inconsistency between which data informed feature selection and which data the model actually trains on.

### The Embargo
The core leakage risk being guarded against: since the target at date `t` is built from returns `t+1..t+5`, a training row at date `T` encodes information about returns as far forward as `T+5`. If a validation period begins immediately at `T+1`, the last several training rows' targets *overlap* the validation period's own realized returns — a genuine information leak between splits, not from the future into the past.

The fix: `drop_last_dates()` removes the last `EMBARGO_DAYS` (=5) **dates** — explicitly not rows — from the end of train (and from val, at the val/test boundary). The distinction between dates and rows is called out because the panel table has ~100 rows per date; dropping 5 rows by position would only remove five individual stocks from a single date, not create the necessary temporal gap.

`make_split()` applies this: `train` and `val` both get their last `EMBARGO_DAYS` dates dropped; `test` is left untouched (nothing follows it that needs protecting from leakage).

## Cell 3 — Validation
An extensive assertion suite run across every `(dataset, split)` combination before anything is saved:

- **No date overlap** between train/val and val/test.
- **Strict temporal ordering** — train's last date must precede val's first date, and val's last date must precede test's first date.
- **Embargo gap sufficiency** — computed via `searchsorted` against the full sorted date index for that dataset, checking the actual number of *trading days* between the end of one role and the start of the next exceeds `HORIZON` (5). This is a stronger check than merely confirming the embargo dropped some dates — it confirms the resulting gap is actually large enough that no training target's forward window can reach into the validation period.
- **No NaN** in features or targets, in any split/part.
- **Clip boundary respected** — max absolute value across non-binary features must not exceed `CLIP` (with a small floating-point tolerance).
- **Column order consistency** — every split/part's column list must exactly match the reference column order from the full (pre-split) dataset, guarding against any accidental column reordering introduced during slicing.

All violations are collected into a single `errors` list and printed together at the end, rather than failing fast on the first problem — so a full audit of every split is visible in one run.

## Cell 4 — Save and Summarize

### Save
Writes each `(dataset, split, part)` combination to its own file under `04_splits/{split_name}/{dataset}_{part}.parquet`, casting all non-key columns to `DTYPE` (float32) — described as "one dtype, straight into torch," i.e. no further conversion needed at model-loading time.

### Split Summary
Builds a summary table across every split/dataset/part combination: row count, unique date count, date range, crash count, crash rate, and mean target value.

**Explicitly calls out crash *count* over crash *rate*** as the more important quantity to watch: AUC computed on a validation set with only ~19 positive events has a standard error near 0.08 — large enough that AUC-based early stopping would be unreliable — which is given as the direct justification for using Brier score (computed on every row, not just positives) for early stopping instead. Validation sets with fewer than 30 crash events are specifically flagged and printed separately as a caution.

### Metadata
Assembles a single `metadata.json` capturing everything a downstream training script needs without re-deriving it: the exact target formulas and their non-z-score rationale, the loss functions (Huber for continuous with per-split delta pulled from `huber_delta.json`, BCEWithLogitsLoss for binary), the early-stopping metric choice and its AUC-instability justification, the full split configuration with embargo days attached, preprocessing constants (clip value, dtype, fill strategy, and the effective start date — noted as "2007-08-01, the latest per-feature warm-up end, CFTC weekly," i.e. the binding constraint on how early usable data begins), and the per-dataset meta-column list and feature count. The full split summary table is embedded as well.

## Diagnostic Cell A — Duplicate Column Check on Unioned Tables
A standalone sanity check (not part of the main pipeline flow) that loads each of the seven Stage 4-unioned tables and checks for any column name appearing more than once — a basic structural integrity check on the outputs of `01_apply_union.ipynb`, run here as a final confirmation before trusting them as merge inputs.

## Diagnostic Cell B — Merge-Suffix and Duplicate Check Across All Assembled and Split Outputs
A broader sweep across every relevant output file — the three assembled tables (`agg_means`, `agg_full_moments`, `panel`) plus every split/part file under `04_splits/` — checking each file's column schema for duplicated names and for the tell-tale `_x`/`_y` suffixes pandas appends automatically when a merge produces a name collision it wasn't told to resolve.

Reads column names via `pyarrow.parquet.read_schema` rather than `pandas.read_parquet`, explicitly because pandas can silently mangle duplicate column names on read (e.g. auto-renaming) — reading the schema directly gets the literal, unmodified column list as actually stored in the file, which is what a check like this needs to be meaningful.

## Output
- `Data/Data_Collection/Final/Stage_5_Model_Ready/04_splits/{Split_A,B,C,D}/{agg_means,agg_full_moments,panel}_{train,val,test}.parquet` — 4 splits × 3 datasets × 3 parts = 36 files.
- `Data/Data_Collection/Final/Stage_5_Model_Ready/04_splits/split_summary.csv`
- `Data/Data_Collection/Final/Stage_5_Model_Ready/04_splits/metadata.json` — full pipeline configuration and target/loss documentation for downstream training code.

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append('../..')
from lib.config import OUT, CLIP, DTYPE, CRASH_THRESH, BINARIES

ASM_DIR = OUT / '02_assembled'
TGT_DIR = OUT / '03_targets'
OUT_DIR = OUT / '04_splits'
OUT_DIR.mkdir(parents=True, exist_ok=True)

HORIZON      = 5
EMBARGO_DAYS = 5     # DATES, not rows -- the panel has ~100 rows per date

# Unchanged from the earlier pipeline. Split_B's train_end is also the
# diagnostic window boundary used to freeze the feature set, so it stays fixed.
SPLITS = {
    'Split_A': {'description': 'Pre-COVID stress (Volmageddon, Q4 2018 selloff)',
                'train_end': '2015-12-31', 'val_start': '2016-01-01',
                'val_end': '2017-12-31', 'test_start': '2018-01-01',
                'test_end': '2019-12-31'},
    'Split_B': {'description': 'Black swan + recovery (COVID crash and rebound)',
                'train_end': '2017-12-31', 'val_start': '2018-01-01',
                'val_end': '2019-12-31', 'test_start': '2020-01-01',
                'test_end': '2021-12-31'},
    'Split_C': {'description': 'Primary: rate hiking bear market',
                'train_end': '2019-12-31', 'val_start': '2020-01-01',
                'val_end': '2021-12-31', 'test_start': '2022-01-01',
                'test_end': '2023-12-31'},
    'Split_D': {'description': 'Recovery + AI rally (most training data)',
                'train_end': '2020-12-31', 'val_start': '2021-01-01',
                'val_end': '2022-12-31', 'test_start': '2023-01-01',
                'test_end': '2024-12-31'},
}

# Everything that is not a feature. Column order in the saved files is
# meta first, then features, so loading is df[feature_cols] with no lookup.
META = {
    'agg_means':        ['date', 'target_daily_return', 'minret_5d_pct', 'y_binary'],
    'agg_full_moments': ['date', 'target_daily_return', 'minret_5d_pct', 'y_binary'],
    'panel':            ['permno', 'date', 'dlyret', 'dlycap',
                         'minret_5d_pct', 'y_binary'],
}

print('=' * 100)
print('LOAD AND MERGE TARGETS')
print('=' * 100)

tgt_m = pd.read_parquet(TGT_DIR / 'targets_market.parquet')
tgt_p = pd.read_parquet(TGT_DIR / 'targets_panel.parquet').drop(columns=['dlyret'])

data = {}
for name in ('agg_means', 'agg_full_moments'):
    f = pd.read_parquet(ASM_DIR / f'{name}.parquet').drop(columns=['target_daily_return'])
    # inner: rows without a complete forward window have no target and drop out
    df = tgt_m.merge(f, on='date', how='inner')
    data[name] = df[META[name] + [c for c in df.columns if c not in META[name]]]

f = pd.read_parquet(ASM_DIR / 'panel.parquet')
df = f.merge(tgt_p, on=['permno', 'date'], how='inner')
data['panel'] = df[META['panel'] + [c for c in df.columns if c not in META['panel']]]

for name, df in data.items():
    n_feat = len(df.columns) - len(META[name])
    print(f'  {name:<18} {len(df):>8,} rows   {n_feat:>5} features   '
          f'{df["date"].min().date()} .. {df["date"].max().date()}')
    assert df[[c for c in df.columns if c not in META[name]]].isna().sum().sum() == 0
del f, df

LOAD AND MERGE TARGETS
  agg_means             4,379 rows     579 features   2007-08-01 .. 2024-12-20
  agg_full_moments      4,379 rows    1704 features   2007-08-01 .. 2024-12-20
  panel               435,329 rows     579 features   2007-08-01 .. 2024-12-20


In [2]:
# ── EMBARGO ──────────────────────────────────────────────────────────────────
# The target at date t is built from returns t+1..t+5, so a training row at
# date T carries information about returns up to T+5. If validation begins at
# T+1, the last five training rows have targets that overlap the validation
# period. Dropping the last EMBARGO_DAYS *dates* from train (and from val, for
# the val/test boundary) removes the overlap exactly.
#
# DATES, not rows. The panel has ~100 rows per date, so dropping 5 rows would
# remove five stocks from one date rather than five days.

def drop_last_dates(df, n):
    if n == 0 or df.empty:
        return df
    keep = np.sort(df['date'].unique())[:-n]
    return df[df['date'].isin(keep)]


def make_split(df, cfg):
    d = df['date']
    return {
        'train': drop_last_dates(df[d <= cfg['train_end']], EMBARGO_DAYS),
        'val':   drop_last_dates(df[(d >= cfg['val_start']) &
                                    (d <= cfg['val_end'])], EMBARGO_DAYS),
        'test':  df[(d >= cfg['test_start']) & (d <= cfg['test_end'])],
    }


print('=' * 100)
print('BUILD SPLITS')
print('=' * 100)

splits = {}
for name, df in data.items():
    for sname, cfg in SPLITS.items():
        parts = make_split(df, cfg)
        splits[(name, sname)] = {k: v.reset_index(drop=True) for k, v in parts.items()}

print(f'  {len(splits)} dataset x split combinations, 3 parts each '
      f'= {len(splits) * 3} files')

BUILD SPLITS
  12 dataset x split combinations, 3 parts each = 36 files


In [3]:
print('=' * 100)
print('VALIDATION')
print('=' * 100)

errors = []
for (name, sname), parts in splits.items():
    tag = f'{sname}/{name}'
    tr, va, te = parts['train'], parts['val'], parts['test']
    feats = [c for c in tr.columns if c not in META[name]]

    for a, b, lbl in [('train', 'val', 'train/val'), ('val', 'test', 'val/test')]:
        overlap = set(parts[a]['date']) & set(parts[b]['date'])
        if overlap:
            errors.append(f'{tag}: {lbl} share {len(overlap)} dates')

    if tr['date'].max() >= va['date'].min():
        errors.append(f'{tag}: train ends at or after val begins')
    if va['date'].max() >= te['date'].min():
        errors.append(f'{tag}: val ends at or after test begins')

    # the embargo must leave at least HORIZON trading days between roles, or a
    # training target still reaches into validation
    all_dates = pd.DatetimeIndex(sorted(data[name]['date'].unique()))
    for a, b, lbl in [(tr, va, 'train->val'), (va, te, 'val->test')]:
        gap = (all_dates.searchsorted(b['date'].min())
               - all_dates.searchsorted(a['date'].max()))
        if gap <= HORIZON:
            errors.append(f'{tag}: only {gap} trading days {lbl}, need > {HORIZON}')

    for part, p in parts.items():
        if p[feats].isna().sum().sum():
            errors.append(f'{tag}/{part}: NaN in features')
        nonbin = [c for c in feats if c not in BINARIES]
        mx = p[nonbin].abs().to_numpy().max()
        if mx > CLIP + 1e-6:
            errors.append(f'{tag}/{part}: max |z| = {mx:.3f} exceeds {CLIP}')
        if p['minret_5d_pct'].isna().any() or p['y_binary'].isna().any():
            errors.append(f'{tag}/{part}: NaN in target')

    ref = list(data[name].columns)
    for part, p in parts.items():
        if list(p.columns) != ref:
            errors.append(f'{tag}/{part}: column order differs from Split_B')

if errors:
    print(f'  {len(errors)} PROBLEM(S):')
    for e in errors:
        print(f'    x {e}')
else:
    print('  all checks passed: no date overlap, temporal order holds, embargo')
    print('  gap exceeds the target horizon, no NaN, clip respected, columns aligned')

VALIDATION
  all checks passed: no date overlap, temporal order holds, embargo
  gap exceeds the target horizon, no NaN, clip respected, columns aligned


In [4]:
print('=' * 100)
print('SAVE')
print('=' * 100)

for (name, sname), parts in splits.items():
    d = OUT_DIR / sname
    d.mkdir(parents=True, exist_ok=True)
    for part, p in parts.items():
        out = p.copy()
        num = [c for c in out.columns if c not in ('date', 'permno')]
        out[num] = out[num].astype(DTYPE)      # one dtype, straight into torch
        out.to_parquet(d / f'{name}_{part}.parquet', index=False)

mb = sum(f.stat().st_size for f in OUT_DIR.rglob('*.parquet')) / 1024**2
print(f'  {len(list(OUT_DIR.rglob("*.parquet")))} files, {mb:,.0f} MB')

# ── SUMMARY ──────────────────────────────────────────────────────────────────
# Crash COUNT matters more than rate: AUC on ~19 positives has a standard error
# near 0.08, which is why early stopping uses Brier score rather than AUC.

print('\n' + '=' * 100)
print('SPLIT SUMMARY')
print('=' * 100)

rows = []
for (name, sname), parts in splits.items():
    for part, p in parts.items():
        rows.append({
            'split': sname, 'dataset': name, 'part': part,
            'rows': len(p), 'dates': p['date'].nunique(),
            'from': p['date'].min().date(), 'to': p['date'].max().date(),
            'crashes': int(p['y_binary'].sum()),
            'crash_rate': float(p['y_binary'].mean()),
            'target_mean': float(p['minret_5d_pct'].mean()),
        })
summary = pd.DataFrame(rows)

for name in data:
    print(f'\n  {name}')
    s = summary[summary['dataset'] == name]
    print(s[['split', 'part', 'rows', 'dates', 'from', 'to', 'crashes',
             'crash_rate', 'target_mean']]
          .to_string(index=False, formatters={'rows': '{:,}'.format,
                                              'crash_rate': '{:.1%}'.format,
                                              'target_mean': '{:.2f}%'.format}))

thin = summary[(summary['part'] == 'val') & (summary['crashes'] < 30)]
if len(thin):
    print('\n  ' + '-' * 96)
    print('  VALIDATION SETS WITH FEW CRASH EVENTS')
    print('  ' + '-' * 96)
    print(thin[['split', 'dataset', 'crashes', 'crash_rate']].to_string(index=False))
    print('\n  Early stopping uses Brier score, which is computed on every')
    print('  validation row rather than on the positives alone.')

summary.to_csv(OUT_DIR / 'split_summary.csv', index=False)

# ── METADATA ─────────────────────────────────────────────────────────────────

with open(TGT_DIR / 'huber_delta.json') as f:
    huber = json.load(f)

meta = {
    'created': pd.Timestamp.now().isoformat(),
    'target': {
        'continuous': f'minret_5d_pct = 100 * min(next {HORIZON} daily returns)',
        'binary':     f'y_binary = minret_5d_pct < {CRASH_THRESH}',
        'horizon_days': HORIZON,
        'crash_threshold_pct': CRASH_THRESH,
        'note': ('raw percentage, not an expanding z-score: a crash is absolute, '
                 'and no z means no shift-by-6 leakage surface'),
    },
    'loss': {
        'continuous': 'Huber, delta per split from huber_delta.json',
        'binary':     'BCEWithLogitsLoss',
        'huber':      huber,
        'early_stopping': {
            'continuous': 'R2',
            'binary':     'Brier score',
            'reason': ('AUC is unstable when the validation window holds few '
           'crash events (Split_A: ~19). Brier uses every row. '
           'Test performance is still reported as AUC.'),
        },
    },
    'splits': {s: {**c, 'embargo_days': EMBARGO_DAYS} for s, c in SPLITS.items()},
    'embargo': {
        'days': EMBARGO_DAYS,
        'applied_to': 'last N DATES of train and of val',
        'reason': ('AUC is unstable when the validation window holds few '
           'crash events (Split_A: ~19). Brier uses every row. '
           'Test performance is still reported as AUC.'),
    },
    'preprocessing': {
        'clip': CLIP,
        'dtype': DTYPE,
        'fill': 'zero after clipping; pre-fill NaN preserved in *_nan.parquet',
        'start_date': '2007-08-01 (latest per-feature warm-up end, CFTC weekly)',
    },
    'datasets': {n: {'meta_cols': META[n],
                     'n_features': len(data[n].columns) - len(META[n])}
                 for n in data},
    'summary': summary.to_dict('records'),
}
with open(OUT_DIR / 'metadata.json', 'w') as f:
    json.dump(meta, f, indent=2, default=str)

print(f'\n  saved -> split_summary.csv, metadata.json')
print(f'\n  load:  pd.read_parquet(OUT/"04_splits"/split/f"{{dataset}}_{{part}}.parquet")')
print(f'         features = [c for c in df.columns if c not in META[dataset]]')

SAVE
  36 files, 1,478 MB

SPLIT SUMMARY

  agg_means
  split  part  rows  dates       from         to  crashes crash_rate target_mean
Split_A train 2,116   2116 2007-08-01 2015-12-23      443      20.9%      -1.29%
Split_A   val   498    498 2016-01-04 2017-12-21       19       3.8%      -0.57%
Split_A  test   503    503 2018-01-02 2019-12-31       83      16.5%      -1.00%
Split_B train 2,619   2619 2007-08-01 2017-12-21      464      17.7%      -1.15%
Split_B   val   498    498 2018-01-02 2019-12-23       83      16.7%      -1.01%
Split_B  test   505    505 2020-01-02 2021-12-31      101      20.0%      -1.36%
Split_C train 3,122   3122 2007-08-01 2019-12-23      547      17.5%      -1.13%
Split_C   val   500    500 2020-01-02 2021-12-23       98      19.6%      -1.36%
Split_C  test   501    501 2022-01-03 2023-12-29      114      22.8%      -1.43%
Split_D train 3,375   3375 2007-08-01 2020-12-23      618      18.3%      -1.18%
Split_D   val   498    498 2021-01-04 2022-12-22      1

In [5]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append('../..')
import lib.review as rv
from lib.config import OUT, META as ASM_META, BINARIES

UNI_DIR = OUT / '01_unioned'


for tag in ['agg_market_daily_means', 'agg_market_daily_full_moments',
            'weekly_raw', 'agg_market_monthly_means',
            'agg_market_monthly_full_moments',
            'panel_stock_daily_engineered', 'panel_stock_monthly_engineered']:
    cols = pd.read_parquet(UNI_DIR / f'{tag}.parquet').columns.tolist()
    dupes = [c for c in set(cols) if cols.count(c) > 1]
    print(f'  {tag:<34} {len(cols):>5} cols, {len(dupes)} duplicated')

  agg_market_daily_means               292 cols, 0 duplicated
  agg_market_daily_full_moments        824 cols, 0 duplicated
  weekly_raw                            33 cols, 0 duplicated
  agg_market_monthly_means             259 cols, 0 duplicated
  agg_market_monthly_full_moments      852 cols, 0 duplicated
  panel_stock_daily_engineered         138 cols, 0 duplicated
  panel_stock_monthly_engineered       157 cols, 0 duplicated


In [6]:
import pyarrow.parquet as pq
from pathlib import Path

OUT = Path('../../../Data/Data_Collection/Final/Stage_5_Model_Ready')
ASM = OUT / '02_assembled'
SPL = OUT / '04_splits'

print('=' * 96)
print('MERGED OUTPUTS - duplicate and merge-suffixed column names')
print('=' * 96)

targets = [(ASM / f'{n}.parquet', n) for n in
           ('agg_means', 'agg_full_moments', 'panel')]
targets += [(p, f'{p.parent.name}/{p.stem}') for p in sorted(SPL.rglob('*.parquet'))]

bad = 0
for path, label in targets:
    if not path.exists():
        print(f'  {label:<44} (not found)')
        continue

    # pq.read_schema, not pd.read_parquet: pandas can mangle duplicate names on
    # read, so the literal list from the file is what you want here.
    cols = pq.read_schema(path).names
    dupes    = sorted({c for c in cols if cols.count(c) > 1})
    suffixed = sorted(c for c in cols if c.endswith(('_x', '_y')))

    flag = ''
    if dupes:
        flag += f'   DUPLICATES: {dupes[:6]}'
    if suffixed:
        flag += f'   MERGE-SUFFIXED: {suffixed[:6]}'
    bad += bool(flag)
    print(f'  {label:<44} {len(cols):>5} cols{flag}')

print(f'\n  {"clean" if bad == 0 else f"{bad} FILE(S) WITH PROBLEMS"}')

MERGED OUTPUTS - duplicate and merge-suffixed column names
  agg_means                                      581 cols
  agg_full_moments                              1706 cols
  panel                                          583 cols
  Split_A/agg_full_moments_test                 1708 cols
  Split_A/agg_full_moments_train                1708 cols
  Split_A/agg_full_moments_val                  1708 cols
  Split_A/agg_means_test                         583 cols
  Split_A/agg_means_train                        583 cols
  Split_A/agg_means_val                          583 cols
  Split_A/panel_test                             585 cols
  Split_A/panel_train                            585 cols
  Split_A/panel_val                              585 cols
  Split_B/agg_full_moments_test                 1708 cols
  Split_B/agg_full_moments_train                1708 cols
  Split_B/agg_full_moments_val                  1708 cols
  Split_B/agg_means_test                         583 cols
  Split_B/agg

In [7]:
import pandas as pd
for s in ['Split_A','Split_B','Split_C','Split_D']:
    d = pd.read_parquet(OUT/'04_splits'/s/'agg_means_train.parquet', columns=['date'])
    print(s, d['date'].max().date())

Split_A 2015-12-23
Split_B 2017-12-21
Split_C 2019-12-23
Split_D 2020-12-23
